In [ ]:
import pyodbc
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import (Font, PatternFill, Alignment, 
                              Border, Side)
from openpyxl.utils import get_column_letter

# ── 1. CONNECTION ──────────────────────────────────────────────
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=DESKTOP-MLRLTD2;"
    "DATABASE=hr_audit;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"  # required for Driver 18
)
print("Connected successfully")

# ── 2. RUN ALL 5 QUERIES ───────────────────────────────────────
q1 = pd.read_sql("""
SELECT
    emp_name,
    COUNT(*)                                                                        AS late_days,
    ROUND(AVG(lateby_mins), 1)                                                      AS avg_late_mins,
    MAX(lateby_mins)                                                                AS worst_late_mins,
    SUM(CASE WHEN late_category = 'Severe - over 1 hour'    THEN 1 ELSE 0 END)     AS severe_count,
    SUM(CASE WHEN late_category = 'Moderate - 30 to 60 min' THEN 1 ELSE 0 END)     AS moderate_count,
    SUM(CASE WHEN late_category = 'Minor - under 30 min'    THEN 1 ELSE 0 END)     AS minor_count
FROM Attendance
WHERE lateby_mins > 0
  AND status_clean IN ('Present', 'Weekend Present')
GROUP BY emp_name
ORDER BY late_days DESC
""", conn)

q2 = pd.read_sql("""
SELECT
    emp_name,
    COUNT(*)                                                                        AS total_days,
    SUM(CASE WHEN status_clean = 'Present'         THEN 1 ELSE 0 END)              AS present_days,
    SUM(CASE WHEN status_clean = 'Absent'          THEN 1 ELSE 0 END)              AS absent_days,
    SUM(CASE WHEN status_clean = 'Weekend Present' THEN 1 ELSE 0 END)              AS weekend_present,
    ROUND(100.0 * SUM(CASE WHEN status_clean = 'Present' THEN 1 ELSE 0 END) /
        NULLIF(SUM(CASE WHEN status_clean
            IN ('Present', 'Absent') THEN 1 ELSE 0 END), 0), 1)                    AS attendance_rate_pct
FROM Attendance
GROUP BY emp_name
ORDER BY attendance_rate_pct ASC
""", conn)

q3 = pd.read_sql("""
SELECT
    emp_name,
    COUNT(*)                            AS ot_days,
    ROUND(SUM(ot_mins)  / 60.0, 2)     AS total_ot_hours,
    ROUND(AVG(ot_mins)  / 60.0, 2)     AS avg_ot_per_day_hrs,
    ROUND(MAX(ot_mins)  / 60.0, 2)     AS max_ot_day_hrs
FROM Attendance
WHERE ot_mins > 0
  AND status_clean IN ('Present', 'Weekend Present')
GROUP BY emp_name
ORDER BY total_ot_hours DESC
""", conn)

q4 = pd.read_sql("""
SELECT
    punch_flag                                                          AS flag_type,
    COUNT(*)                                                            AS record_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM Attendance), 1)     AS pct_of_total
FROM Attendance
GROUP BY punch_flag
ORDER BY record_count DESC
""", conn)

q5 = pd.read_sql("""
SELECT
    emp_name,
    SUM(CASE WHEN status_clean = 'Absent' THEN 1 ELSE 0 END)               AS absent_days,
    SUM(CASE WHEN late_category = 'Severe - over 1 hour'
        THEN 1 ELSE 0 END)                                                  AS severe_late_days,
    SUM(CASE WHEN late_category IN (
        'Severe - over 1 hour',
        'Moderate - 30 to 60 min') THEN 1 ELSE 0 END)                      AS total_late_days,
    ROUND(100.0 * SUM(CASE WHEN status_clean = 'Present' THEN 1 ELSE 0 END) /
        NULLIF(SUM(CASE WHEN status_clean
            IN ('Present', 'Absent') THEN 1 ELSE 0 END), 0), 1)            AS attendance_rate_pct,
    CASE
        WHEN SUM(CASE WHEN status_clean = 'Absent'
            THEN 1 ELSE 0 END) >= 15 THEN 'Critical - immediate review'
        WHEN SUM(CASE WHEN status_clean = 'Absent'
            THEN 1 ELSE 0 END) >= 10 THEN 'High - manager follow-up'
        WHEN SUM(CASE WHEN late_category = 'Severe - over 1 hour'
            THEN 1 ELSE 0 END) >= 3  THEN 'Moderate - coaching needed'
        ELSE 'Monitor'
    END                                                                     AS hr_action
FROM Attendance
GROUP BY emp_name
HAVING
    SUM(CASE WHEN status_clean = 'Absent' THEN 1 ELSE 0 END) >= 5
    OR SUM(CASE WHEN late_category = 'Severe - over 1 hour'
        THEN 1 ELSE 0 END) >= 3
ORDER BY absent_days DESC, severe_late_days DESC
""", conn)

conn.close()

Connected successfully
All 5 queries complete
Q1: 25 rows | Q2: 29 rows | Q3: 23 rows | Q4: 6 rows | Q5: 29 rows


C:\Users\honey\AppData\Local\Temp\ipykernel_7452\2417206832.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q1 = pd.read_sql("""
C:\Users\honey\AppData\Local\Temp\ipykernel_7452\2417206832.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q2 = pd.read_sql("""
C:\Users\honey\AppData\Local\Temp\ipykernel_7452\2417206832.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q3 = pd.read_sql("""
C:\Users\honey\AppData\Local\Temp\ipykernel_7452\2417206832.py:64: UserWarning: pandas only supports SQLAlchemy connectable (en

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── HELPER FUNCTIONS ───────────────────────────────────────────
def style_header(ws, row):
    header_fill = PatternFill("solid", fgColor="1A3C5E")
    header_font = Font(bold=True, color="FFFFFF", size=11)
    for cell in ws[row]:
        if cell.value is not None:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center")

def style_table(ws, num_rows, start_row):
    thin = Side(style="thin", color="CCCCCC")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill_alt = PatternFill("solid", fgColor="EBF2FA")
    for row_idx, row in enumerate(
        ws.iter_rows(min_row=start_row, max_row=start_row + num_rows - 1), start=0
    ):
        for cell in row:
            cell.border = border
            cell.alignment = Alignment(horizontal="center", vertical="center")
            if row_idx % 2 == 1:
                cell.fill = fill_alt

def autofit_columns(ws):
    for col in ws.columns:
        max_len = max(
            (len(str(cell.value)) for cell in col if cell.value is not None),
            default=10
        )
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 4, 40)

def write_title(ws, title, subtitle):
    """Write title into rows 1 and 2 directly — no insert_rows"""
    ws["A1"] = title
    ws["A1"].font = Font(bold=True, size=14, color="1A3C5E")
    ws["A2"] = subtitle
    ws["A2"].font = Font(italic=True, size=10, color="555555")


# ── BUILD WORKBOOK ─────────────────────────────────────────────
# startrow=2 means: row 1=title, row 2=subtitle, row 3=header, row 4+=data
output_path = r"..\data\HR_Audit_Report.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    q1.to_excel(writer, sheet_name="Late Arrivals",      index=False, startrow=2)
    q2.to_excel(writer, sheet_name="Absenteeism",        index=False, startrow=2)
    q3.to_excel(writer, sheet_name="Overtime",           index=False, startrow=2)
    q4.to_excel(writer, sheet_name="Punch Integrity",    index=False, startrow=2)
    q5.to_excel(writer, sheet_name="HR Action Required", index=False, startrow=2)

wb = load_workbook(output_path)

sheets = {
    "Late Arrivals"     : (q1, "Late Arrival Summary — March 2025",
                           "Days late, average lateness, and severity breakdown per employee"),
    "Absenteeism"       : (q2, "Absenteeism Report — March 2025",
                           "Attendance rate and absent day count per employee"),
    "Overtime"          : (q3, "Overtime Summary — March 2025",
                           "Total, average, and peak overtime hours per employee"),
    "Punch Integrity"   : (q4, "Punch Record Integrity Audit — March 2025",
                           "Breakdown of biometric punch record quality across all records"),
    "HR Action Required": (q5, "Chronic Attendance Issues — March 2025",
                           "Employees flagged for HR review based on absence and lateness patterns"),
}

for sheet_name, (df, title, subtitle) in sheets.items():
    ws = wb[sheet_name]

    # Write title directly into rows 1 and 2
    write_title(ws, title, subtitle)

    # Header is at row 3, data starts at row 4
    style_header(ws, row=3)
    style_table(ws, num_rows=len(df), start_row=4)
    autofit_columns(ws)

    # Color code HR Action Required sheet
    if sheet_name == "HR Action Required":
        action_col_idx = q5.columns.get_loc("hr_action") + 1
        colors = {
            "Critical - immediate review" : "FFDDDD",
            "High - manager follow-up"    : "FFE8CC",
            "Moderate - coaching needed"  : "FFF9CC",
            "Monitor"                     : "E8F5E9",
        }
        for row in ws.iter_rows(min_row=4, max_row=3 + len(q5)):
            cell = row[action_col_idx - 1]
            fill_color = colors.get(str(cell.value), "FFFFFF")
            cell.fill = PatternFill("solid", fgColor=fill_color)
            cell.font = Font(bold=True, size=10)

wb.save(output_path)
print(f"Exported: {output_path}")

Exported: ..\data\HR_Audit_Report.xlsx
Rows per sheet — Q1:25 Q2:29 Q3:23 Q4:6 Q5:29
